# 03 — Gold Aggregation

Reads Silver tables via **Change Data Feed** and writes four Gold tables:

| Table | Write mode | MERGE key | Upsert scenario |
|---|---|---|---|
| `gold_driver_championship` | **MERGE** | (season, driver_id) | Updated in-place after every round |
| `gold_constructor_championship` | **MERGE** | (season, constructor_id) | Updated in-place after every round |
| `gold_circuit_benchmarks` | **MERGE** | circuit_id | Conditional update when faster lap set |
| `gold_tyre_strategy_report` | APPEND | — | One row per driver per race |

A **Delta Time Travel** demo cell at the end shows how to query
`gold_driver_championship` as it stood after any historical race date.

In [ ]:
%pip install /Workspace/Users/pravbala30@protonmail.com/.bundle/f1-intelligence/dev/artifacts/.internal/f1_intelligence-0.1.0-py3-none-any.whl --quiet
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "f1_intelligence")
dbutils.widgets.text("schema",  "f1_dev")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

print(f"catalog={CATALOG}  schema={SCHEMA}")

In [ ]:
# Setup
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql.types import IntegerType, DoubleType
from datetime import datetime, timezone

from utils.helpers import (
    get_or_create_catalog_schema,
    read_incremental_or_full,
    merge_delta,
    write_delta_append,
    save_checkpoint,
    get_current_table_version,
    print_table_stats,
    table_exists,
)
from utils.schema import MERGE_KEYS
from utils.validators import validate_gold_standings

get_or_create_catalog_schema(spark, CATALOG, SCHEMA)

def T(table): return f"{CATALOG}.{SCHEMA}.{table}"
CKPT = T("pipeline_checkpoints")

print("Setup complete")


In [ ]:
# Read Silver changes since last checkpoint
df_silver_results, silver_results_version = read_incremental_or_full(
    spark, T("silver_race_results"), CKPT, "silver_to_gold"
)

if df_silver_results.isEmpty():
    print("No new Silver data — nothing to aggregate. Exiting.")
    dbutils.notebook.exit("no_new_data")

# Always read full Silver standings (needed for cumulative aggregation)
df_silver_results_full = spark.read.format("delta").table(T("silver_race_results"))
df_silver_standings    = spark.read.format("delta").table(T("silver_driver_standings"))
df_silver_cs           = spark.read.format("delta").table(T("silver_constructor_standings"))
df_silver_qual         = spark.read.format("delta").table(T("silver_qualifying")) \
    if table_exists(spark, T("silver_qualifying")) else None

print(f"New Silver race_results rows: {df_silver_results.count():,}")

In [ ]:
# ── Gold Driver Championship (MERGE — 1 row per driver per season) ──────────
print("Building gold_driver_championship...")

# Latest round standings snapshot per season+driver
latest_round_window = Window.partitionBy("season", "driver_id").orderBy(F.col("round").desc())
df_latest_standings = (
    df_silver_standings
    .withColumn("rn", F.row_number().over(latest_round_window))
    .filter(F.col("rn") == 1)
    .select("season", "driver_id", "driver_code", "constructor_id",
            F.col("position").alias("current_position"),
            F.col("points").alias("current_points"),
            "wins",
            F.col("round").alias("last_round_processed"))
)

# Aggregate per-race stats from full race results
df_race_stats = (
    df_silver_results_full
    .groupBy("season", "driver_id")
    .agg(
        F.sum(F.when(F.col("final_position").between(1, 3), 1).otherwise(0)).alias("podiums"),
        F.sum(F.when(~F.col("is_classified"), 1).otherwise(0)).alias("dnfs"),
        F.count("*").alias("races_completed"),
    )
)

# Best qualifying position
if df_silver_qual is not None:
    df_best_qual = (
        df_silver_qual
        .groupBy("season", "driver_id")
        .agg(F.min("qualifying_position").alias("best_qualifying_position"))
    )
else:
    df_best_qual = df_latest_standings.select("season", "driver_id").withColumn(
        "best_qualifying_position", F.lit(None).cast(IntegerType())
    )

df_driver_champ = (
    df_latest_standings
    .join(df_race_stats,  on=["season", "driver_id"], how="left")
    .join(df_best_qual,   on=["season", "driver_id"], how="left")
    .withColumn("updated_at", F.current_timestamp())
    .select(
        "season", "driver_id", "driver_code", "constructor_id",
        "current_position", "current_points", "wins", "podiums", "dnfs",
        "best_qualifying_position", "races_completed", "last_round_processed",
        "updated_at",
    )
)

result = validate_gold_standings(df_driver_champ, "gold_driver_championship")
result.raise_if_failed()

merge_delta(spark, df_driver_champ, T("gold_driver_championship"),
            MERGE_KEYS["gold_driver_championship"])
print_table_stats(spark, T("gold_driver_championship"))
print("gold_driver_championship done")


In [ ]:
# ── Gold Constructor Championship (MERGE — 1 row per constructor per season) ─
print("Building gold_constructor_championship...")

latest_cs_window = Window.partitionBy("season", "constructor_id").orderBy(F.col("round").desc())
df_latest_cs = (
    df_silver_cs
    .withColumn("rn", F.row_number().over(latest_cs_window))
    .filter(F.col("rn") == 1)
    .select("season", "constructor_id", "constructor_name",
            F.col("position").alias("current_position"),
            F.col("points").alias("current_points"),
            "wins",
            F.col("round").alias("last_round_processed"))
)

df_cs_stats = (
    df_silver_results_full
    .groupBy("season", "constructor_id")
    .agg(
        F.sum(F.when(F.col("final_position").between(1, 3), 1).otherwise(0)).alias("podiums"),
        F.count("*").alias("races_completed"),
    )
)

df_constructor_champ = (
    df_latest_cs
    .join(df_cs_stats, on=["season", "constructor_id"], how="left")
    .withColumn("updated_at", F.current_timestamp())
    .select(
        "season", "constructor_id", "constructor_name",
        "current_position", "current_points", "wins", "podiums",
        "races_completed", "last_round_processed", "updated_at",
    )
)

result = validate_gold_standings(df_constructor_champ, "gold_constructor_championship")
result.raise_if_failed()

merge_delta(spark, df_constructor_champ, T("gold_constructor_championship"),
            MERGE_KEYS["gold_constructor_championship"])
print_table_stats(spark, T("gold_constructor_championship"))
print("gold_constructor_championship done")


In [ ]:
# ── Gold Circuit Benchmarks (conditional MERGE — update only on faster lap) ──
print("Building gold_circuit_benchmarks...")

df_schedule = spark.read.format("delta").table(T("bronze_race_schedule")) \
    .select("season", "round", "circuit_name", "country")

# Fastest race lap per circuit (from silver_race_results)
df_fastest_lap = (
    df_silver_results_full
    .filter(F.col("fastest_lap_seconds").isNotNull() & (F.col("fastest_lap_rank") == 1))
    .join(df_schedule, on=["season", "round"], how="left")
    .groupBy("circuit_id")
    .agg(
        F.first("circuit_name").alias("circuit_name"),
        F.first("country").alias("country"),
        F.min("fastest_lap_seconds").alias("all_time_fastest_lap_seconds"),
    )
)

# Find the driver/season/round that set the all-time fastest lap
df_fastest_lap_detail = (
    df_silver_results_full
    .filter(F.col("fastest_lap_seconds").isNotNull() & (F.col("fastest_lap_rank") == 1))
    .join(df_schedule, on=["season", "round"], how="left")
)

fastest_lap_window = Window.partitionBy("circuit_id").orderBy(F.col("fastest_lap_seconds"))
df_fastest_lap_who = (
    df_fastest_lap_detail
    .withColumn("rn", F.row_number().over(fastest_lap_window))
    .filter(F.col("rn") == 1)
    .select(
        "circuit_id",
        F.col("driver_id").alias("fastest_lap_driver_id"),
        F.col("constructor_id").alias("fastest_lap_constructor_id"),
        F.col("season").alias("fastest_lap_season"),
        F.col("round").alias("fastest_lap_round"),
    )
)

# Pole position best per circuit (from silver_qualifying)
if df_silver_qual is not None:
    df_pole = (
        df_silver_qual
        .filter(F.col("qualifying_position") == 1)
        .groupBy("circuit_id")
        .agg(F.min("best_qualifying_time_seconds").alias("all_time_pole_seconds"))
    )
    pole_window = Window.partitionBy("circuit_id").orderBy(F.col("best_qualifying_time_seconds"))
    df_pole_who = (
        df_silver_qual
        .filter(F.col("qualifying_position") == 1)
        .withColumn("rn", F.row_number().over(pole_window))
        .filter(F.col("rn") == 1)
        .select(
            "circuit_id",
            F.col("driver_id").alias("pole_driver_id"),
            F.col("season").alias("pole_season"),
            F.col("round").alias("pole_round"),
        )
    )
else:
    df_pole     = df_fastest_lap.select("circuit_id").withColumn("all_time_pole_seconds", F.lit(None).cast(DoubleType()))
    df_pole_who = df_fastest_lap.select("circuit_id").withColumn("pole_driver_id", F.lit(None)) \
        .withColumn("pole_season", F.lit(None).cast(IntegerType())) \
        .withColumn("pole_round",  F.lit(None).cast(IntegerType()))

# Last race winner per circuit
last_race_window = Window.partitionBy("circuit_id").orderBy(F.col("season").desc(), F.col("round").desc())
df_last_winner = (
    df_silver_results_full
    .filter(F.col("final_position") == 1)
    .join(df_schedule, on=["season", "round"], how="left")
    .withColumn("rn", F.row_number().over(last_race_window))
    .filter(F.col("rn") == 1)
    .select(
        "circuit_id",
        F.col("driver_id").alias("last_race_winner_id"),
        F.col("season").alias("last_race_season"),
        F.col("round").alias("last_race_round"),
    )
)

# Total races held per circuit
df_race_count = (
    df_silver_results_full
    .join(df_schedule, on=["season", "round"], how="left")
    .select("circuit_id", "season", "round").distinct()
    .groupBy("circuit_id")
    .agg(F.count("*").alias("total_races_held"))
)

df_circuit_benchmarks = (
    df_fastest_lap
    .join(df_fastest_lap_who, on="circuit_id", how="left")
    .join(df_pole,            on="circuit_id", how="left")
    .join(df_pole_who,        on="circuit_id", how="left")
    .join(df_last_winner,     on="circuit_id", how="left")
    .join(df_race_count,      on="circuit_id", how="left")
    .withColumn("updated_at", F.current_timestamp())
    .select(
        "circuit_id", "circuit_name", "country",
        "all_time_fastest_lap_seconds",
        "fastest_lap_driver_id", "fastest_lap_constructor_id",
        "fastest_lap_season", "fastest_lap_round",
        "all_time_pole_seconds", "pole_driver_id", "pole_season", "pole_round",
        "total_races_held",
        "last_race_winner_id", "last_race_season", "last_race_round",
        "updated_at",
    )
)

merge_delta(spark, df_circuit_benchmarks, T("gold_circuit_benchmarks"),
            MERGE_KEYS["gold_circuit_benchmarks"])
print_table_stats(spark, T("gold_circuit_benchmarks"))
print("gold_circuit_benchmarks done")



In [ ]:
# ── Gold Tyre Strategy Report (APPEND per race) ─────────────────────────────
print("Building gold_tyre_strategy_report...")

# Use only the new Silver lap data in this incremental batch
if not table_exists(spark, T("silver_lap_analysis")):
    print("silver_lap_analysis not yet available — skipping tyre strategy")
else:
    # Determine which (season, round) pairs are new in this batch
    new_rounds = (
        df_silver_results
        .select("season", "round").distinct()
        .collect()
    )

    if new_rounds:
        # Only process laps for the new rounds
        new_rounds_filter = F.col("round").isin([r["round"] for r in new_rounds]) & \
                            F.col("season").isin([r["season"] for r in new_rounds])

        df_laps = spark.read.format("delta").table(T("silver_lap_analysis")) \
            .filter(new_rounds_filter)
        df_schedule = spark.read.format("delta").table(T("bronze_race_schedule")) \
            .select("season", "round", "circuit_id", F.to_date("race_date", "yyyy-MM-dd").alias("race_date"))

        # Pivot stints: up to 3 stints per driver per race
        df_stint_pivot = (
            df_laps
            .filter(F.col("stint_number").isNotNull())
            .groupBy("season", "round", "driver_id", "driver_code", "constructor_id", "stint_number", "compound")
            .agg(F.count("lap_number").alias("stint_laps"))
        )

        # Aggregate per driver per race
        df_driver_race_stints = (
            df_stint_pivot
            .groupBy("season", "round", "driver_id", "driver_code", "constructor_id")
            .agg(
                F.count("stint_number").alias("total_stints"),
                F.concat_ws(",", F.collect_list("compound")).alias("compounds_used"),
                F.first(F.when(F.col("stint_number") == 1, F.col("compound"))).alias("stint_1_compound"),
                F.first(F.when(F.col("stint_number") == 1, F.col("stint_laps"))).alias("stint_1_laps"),
                F.first(F.when(F.col("stint_number") == 2, F.col("compound"))).alias("stint_2_compound"),
                F.first(F.when(F.col("stint_number") == 2, F.col("stint_laps"))).alias("stint_2_laps"),
                F.first(F.when(F.col("stint_number") == 3, F.col("compound"))).alias("stint_3_compound"),
                F.first(F.when(F.col("stint_number") == 3, F.col("stint_laps"))).alias("stint_3_laps"),
            )
        )

        # Join race results for final_position
        df_tyre_strategy = (
            df_driver_race_stints
            .join(
                df_silver_results_full.select("season", "round", "driver_id", "final_position"),
                on=["season", "round", "driver_id"], how="left"
            )
            .join(df_schedule, on=["season", "round"], how="left")
            .withColumn("pit_count", F.col("total_stints") - 1)
            .withColumn("total_pit_time_seconds", F.lit(None).cast(DoubleType()))  # pit duration from pit_stops table if available
            .withColumn("processed_at", F.current_timestamp())
            .select(
                "season", "round", "circuit_id", "race_date",
                "driver_id", "driver_code", "constructor_id", "final_position",
                "total_stints", "compounds_used",
                "stint_1_compound", "stint_1_laps",
                "stint_2_compound", "stint_2_laps",
                "stint_3_compound", "stint_3_laps",
                "total_pit_time_seconds", "pit_count",
                "processed_at",
            )
        )

        if not table_exists(spark, T("gold_tyre_strategy_report")):
            df_tyre_strategy.write.format("delta").mode("overwrite").saveAsTable(T("gold_tyre_strategy_report"))
        else:
            write_delta_append(df_tyre_strategy, T("gold_tyre_strategy_report"))

        print_table_stats(spark, T("gold_tyre_strategy_report"))
        print("gold_tyre_strategy_report done")
    else:
        print("No new rounds — skipping tyre strategy append")



In [ ]:
# ── Save Silver→Gold checkpoint ─────────────────────────────────────────────
save_checkpoint(
    spark, CKPT, "silver_to_gold",
    processed_version=silver_results_version,
    records_processed=df_silver_results.count(),
)

print("\n=== Gold aggregation complete ===")
for t in ["gold_driver_championship", "gold_constructor_championship",
           "gold_circuit_benchmarks", "gold_tyre_strategy_report"]:
    if table_exists(spark, T(t)):
        print_table_stats(spark, T(t))

## Delta Time Travel Demo

Each `MERGE` into `gold_driver_championship` creates a new Delta commit.
We can query the table as it stood after any race by using `TIMESTAMP AS OF`
with the race date — this is robust to re-runs, unlike `VERSION AS OF`.

> **How it works**: After the 2024 Bahrain GP (March 2 2024), the first MERGE
> writes version 0 of the Gold table. After Saudi Arabia (March 9 2024),
> a second MERGE updates standings — version 1. `TIMESTAMP AS OF '2024-03-05'`
> returns the state between those two versions, i.e. the Bahrain standings.

In [ ]:
# ── Delta Time Travel: championship standings after Bahrain 2024 ────────────

# Option A: TIMESTAMP AS OF (recommended — robust to re-runs)
bahrain_2024_race_date = "2024-03-03"  # day after Bahrain GP

print(f"Championship standings as of {bahrain_2024_race_date} (after Bahrain GP):")
try:
    df_after_bahrain = spark.sql(f"""
        SELECT driver_code, constructor_id, current_position, current_points, wins
        FROM {T('gold_driver_championship')}
        TIMESTAMP AS OF '{bahrain_2024_race_date}'
        WHERE season = 2024
        ORDER BY current_position
    """)
    display(df_after_bahrain)
except Exception as e:
    print(f"Note: Time Travel not available yet (table may not have history older than {bahrain_2024_race_date})")
    print(f"Current standings:")
    display(spark.sql(f"""
        SELECT driver_code, constructor_id, current_position, current_points, wins
        FROM {T('gold_driver_championship')}
        WHERE season = 2024
        ORDER BY current_position
    """))

# Option B: Full season history via silver_driver_standings (for trend charts)
print("\nFull championship progression — all rounds:")
display(spark.sql(f"""
    SELECT season, round, driver_code, position, points, position_change
    FROM {T('silver_driver_standings')}
    WHERE season = 2024
    ORDER BY round, position
"""))

dbutils.notebook.exit("gold_aggregation_complete")